# arXiv → SPECTER2 Embedding + FAISS Index (optimized, resumable, up to 2 GPUs)

**Chế độ hiện tại: `SKIP_EMBEDDING = True`.** Notebook tái sử dụng embedding shards đã tính, rồi validate + build FAISS + smoke test. Không load SPECTER2 paper encoder và không chạy lại pipeline embed.

Notebook này tạo **offline dense corpus** cho Research Assistance Agent:

`arXiv metadata snapshot → filter → SPECTER2 proximity embedding → normalized embedding shards → metadata shards → FAISS IndexFlatIP`

Các thay đổi chính so với bản đầu:
- dùng metadata `versions[0].created` thay vì suy năm chủ yếu từ arXiv ID;
- giữ metadata cần cho citation/bibliography và `content_hash` cho incremental update sau này;
- không nhét embedding vào giant pandas DataFrame;
- checkpoint/resume thật bằng `manifest.json`;
- tự detect và dùng tối đa **2 GPU**, mỗi GPU một model replica;
- FP16 autocast trên CUDA + OOM fallback bằng cách chia nhỏ sub-batch;
- L2-normalize embedding trước khi lưu/index;
- build FAISS theo từng shard, không merge toàn corpus vào RAM;
- smoke test retrieval bằng **SPECTER2 adhoc-query adapter**.

> Khi **bỏ qua embedding**: Add Input artifacts từ lần chạy trước (thư mục có `manifest.json`, `embeddings/`, `metadata/`). Snapshot arXiv không bắt buộc. GPU vẫn hữu ích cho smoke test query encoder, nhưng không còn cần 2 GPU để embed.
>
> Khi **chạy embedding**: Add Input dataset `Cornell-University/arxiv`, bật GPU. Với Kaggle T4 x2 notebook sẽ tự dùng cả hai GPU.

**Khuyến nghị:** lần đầu đặt `MAX_PAPERS = 5000` để smoke test toàn pipeline. Khi ổn, đổi thành `None` và chạy full. `MAX_PAPERS` không nằm trong corpus fingerprint nên có thể tiếp tục từ checkpoint nếu artifacts vẫn còn.

## 1. Cài đặt

In [9]:
# Do NOT upgrade Kaggle core packages such as pandas/pyarrow here.
# The previous `-U ... pandas ...` can upgrade pandas to 3.x and create conflicts
# with RAPIDS / BigFrames / Gradio already installed in the Kaggle image.
!pip install -q "adapters>=1.2,<2" sentencepiece faiss-cpu


## 2. Imports + cấu hình

In [10]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

import gc
import json
import math
import time
import hashlib
import platform
import shutil
import importlib.metadata as importlib_metadata
from pathlib import Path
from email.utils import parsedate_to_datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm


# ---------- Skip embedding (reuse existing artifacts) ----------
# True: không load paper encoder, không embed lại. Chỉ validate + FAISS + smoke test.
SKIP_EMBEDDING = True

# Thư mục đã có manifest.json + embeddings/ + metadata/.
# New Kaggle session: Add Input output lần embed trước, rồi trỏ vào đây.
# None = tự tìm trong OUTPUT_DIR hoặc /kaggle/input/*/
EMBEDDING_INPUT_DIR = None  # ví dụ: Path("/kaggle/input/specter2-artifacts")

# ---------- Input / output ----------
SNAPSHOT_PATH = "/kaggle/input/arxiv/arxiv-metadata-oai-snapshot.json"
OUTPUT_DIR = Path("/kaggle/input/datasets/thanhtruchhong/specter2-artifacts/specter2_artifacts")
METADATA_DIR = OUTPUT_DIR / "metadata"
EMBEDDING_DIR = OUTPUT_DIR / "embeddings"
INDEX_DIR = OUTPUT_DIR / "index"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"
INDEX_PATH = INDEX_DIR / "papers_flatip.faiss"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)
if not SKIP_EMBEDDING:
    METADATA_DIR.mkdir(parents=True, exist_ok=True)
    EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Corpus scope ----------
TARGET_CATEGORIES = {"cs.LG", "cs.AI", "cs.CL", "cs.CV", "stat.ML"}
MIN_YEAR = 2021

# None = full corpus after filter. Set 5000 for the first smoke test.
MAX_PAPERS = None

# ---------- SPECTER2 ----------
BASE_MODEL = "allenai/specter2_base"
PAPER_ADAPTER = "allenai/specter2"            # proximity / candidate-paper encoder
QUERY_ADAPTER = "allenai/specter2_adhoc_query" # short textual query encoder
MAX_LENGTH = 512

# ---------- Throughput ----------
MAX_GPUS = 2
BATCH_SIZE_PER_GPU = 32
CPU_BATCH_SIZE = 8
USE_FP16 = True
QUERY_GPU = 0  # dedicated adhoc-query encoder; falls back to CPU when CUDA is unavailable
SHARD_SIZE = 10_000  # checkpoint granularity, papers/shard
STORE_DTYPE = np.float16

# Resume is safe only when corpus config + snapshot fingerprint match.
RESUME = True

print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("SKIP_EMBEDDING:", SKIP_EMBEDDING)
print("Snapshot:", SNAPSHOT_PATH)

Python: 3.12.13
Torch: 2.10.0+cu128
SKIP_EMBEDDING: True
Snapshot: /kaggle/input/arxiv/arxiv-metadata-oai-snapshot.json


## 3. Runtime preflight + reproducibility fingerprint + reuse artifacts

In [11]:
if SKIP_EMBEDDING:
    print("SKIP_EMBEDDING=True -> không yêu cầu arXiv snapshot")
else:
    assert os.path.exists(SNAPSHOT_PATH), (
        f"Không tìm thấy {SNAPSHOT_PATH}. Hãy Add Input Cornell-University/arxiv trước."
    )

CUDA_AVAILABLE = torch.cuda.is_available()
GPU_COUNT = torch.cuda.device_count() if CUDA_AVAILABLE else 0
NUM_DEVICES = min(MAX_GPUS, GPU_COUNT) if GPU_COUNT else 0
DEVICES = [torch.device(f"cuda:{i}") for i in range(NUM_DEVICES)] or [torch.device("cpu")]

if CUDA_AVAILABLE:
    for i in range(GPU_COUNT):
        prop = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {prop.name} | VRAM={prop.total_memory / 1024**3:.1f} GB")
else:
    print("CUDA unavailable -> CPU fallback")

print(f"Devices: {DEVICES}")

# Helps newer NVIDIA GPUs; harmless on T4 where TF32 is not supported.
if CUDA_AVAILABLE:
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

CORPUS_CONFIG = {
    "target_categories": sorted(TARGET_CATEGORIES),
    "min_year": MIN_YEAR,
    "base_model": BASE_MODEL,
    "paper_adapter": PAPER_ADAPTER,
    "max_length": MAX_LENGTH,
    "store_dtype": np.dtype(STORE_DTYPE).name,
}

def stable_hash(obj) -> str:
    payload = json.dumps(obj, sort_keys=True, ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

CONFIG_FINGERPRINT = stable_hash(CORPUS_CONFIG)
if os.path.exists(SNAPSHOT_PATH):
    SNAPSHOT_FINGERPRINT = {
        "basename": os.path.basename(SNAPSHOT_PATH),
        "size_bytes": os.path.getsize(SNAPSHOT_PATH),
    }
    print("Snapshot size (GB):", round(SNAPSHOT_FINGERPRINT["size_bytes"] / 1024**3, 3))
else:
    SNAPSHOT_FINGERPRINT = None
    print("Snapshot: (not loaded)")

print("Config fingerprint:", CONFIG_FINGERPRINT[:16])


def _has_manifest(path: Path) -> bool:
    return path.is_dir() and (path / "manifest.json").exists()


def find_existing_artifacts() -> Path:
    """Locate a completed embedding run: manifest.json + embeddings/ + metadata/."""
    if EMBEDDING_INPUT_DIR is not None:
        p = Path(EMBEDDING_INPUT_DIR)
        if _has_manifest(p):
            return p
        raise FileNotFoundError(f"EMBEDDING_INPUT_DIR không có manifest.json: {p}")

    if _has_manifest(OUTPUT_DIR):
        return OUTPUT_DIR

    found = []
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for slug in sorted(input_root.iterdir()):
            if not slug.is_dir():
                continue
            for cand in (slug, slug / "specter2_artifacts"):
                if _has_manifest(cand) and cand not in found:
                    found.append(cand)
            try:
                children = list(slug.iterdir())
            except OSError:
                children = []
            for child in children:
                if child.is_dir() and _has_manifest(child) and child not in found:
                    found.append(child)

    if not found:
        raise FileNotFoundError(
            "SKIP_EMBEDDING=True nhưng không tìm thấy manifest.json. "
            "Hãy Add Input artifacts từ lần embed trước, hoặc set EMBEDDING_INPUT_DIR."
        )
    if len(found) > 1:
        print("Tìm thấy nhiều thư mục artifacts, dùng:", found[0])
        for extra in found[1:]:
            print("  (bỏ qua)", extra)
    return found[0]


def link_or_reuse_artifacts(src: Path):
    """Expose input shards via OUTPUT_DIR (symlink) so later cells stay unchanged."""
    src = src.resolve()
    dst = OUTPUT_DIR.resolve()
    if src == dst:
        print("Artifacts already in OUTPUT_DIR:", dst)
        return

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    INDEX_DIR.mkdir(parents=True, exist_ok=True)

    for name in ("embeddings", "metadata"):
        src_path = src / name
        dst_path = OUTPUT_DIR / name
        if not src_path.exists():
            raise FileNotFoundError(src_path)
        if dst_path.exists() or dst_path.is_symlink():
            print(f"Keep existing {dst_path}")
            continue
        os.symlink(src_path, dst_path)
        print(f"Linked {name}/ -> {src_path}")

    if not MANIFEST_PATH.exists():
        shutil.copy2(src / "manifest.json", MANIFEST_PATH)
        print("Copied manifest.json ->", MANIFEST_PATH)

    src_index = src / "index" / INDEX_PATH.name
    if src_index.exists() and not INDEX_PATH.exists():
        shutil.copy2(src_index, INDEX_PATH)
        print("Copied existing FAISS index ->", INDEX_PATH)


if SKIP_EMBEDDING:
    artifacts_src = find_existing_artifacts()
    print("Reusing embedding artifacts from:", artifacts_src)
    link_or_reuse_artifacts(artifacts_src)

SKIP_EMBEDDING=True -> không yêu cầu arXiv snapshot
GPU 0: Tesla T4 | VRAM=14.6 GB
GPU 1: Tesla T4 | VRAM=14.6 GB
Devices: [device(type='cuda', index=0), device(type='cuda', index=1)]
Snapshot: (not loaded)
Config fingerprint: 72350168c8d3570b
Reusing embedding artifacts from: /kaggle/input/datasets/thanhtruchhong/specter2-artifacts/specter2_artifacts
Artifacts already in OUTPUT_DIR: /kaggle/input/datasets/thanhtruchhong/specter2-artifacts/specter2_artifacts


## 4. Metadata normalization + streaming filter

Giữ helper definitions (rẻ). Chỉ *gọi* khi `SKIP_EMBEDDING = False`.

In [12]:
def parse_arxiv_id_year(arxiv_id: str) -> int:
    """Fallback only. Modern arXiv IDs start with YYMM."""
    try:
        core = arxiv_id.split("/")[-1]
        yy = int(core[:2])
        return 2000 + yy if yy < 50 else 1900 + yy
    except Exception:
        return 0


def parse_created(created: str):
    if not created:
        return None
    try:
        return parsedate_to_datetime(created)
    except Exception:
        return None


def normalize_record(rec: dict):
    arxiv_id = (rec.get("id") or "").strip()
    title = " ".join((rec.get("title") or "").split())
    abstract = " ".join((rec.get("abstract") or "").split())
    if not arxiv_id or not title or not abstract:
        return None

    all_categories = set((rec.get("categories") or "").split())
    matched = sorted(all_categories & TARGET_CATEGORIES)
    if not matched:
        return None

    versions = rec.get("versions") or []
    first_dt = parse_created(versions[0].get("created")) if versions else None
    latest_dt = parse_created(versions[-1].get("created")) if versions else None

    year = first_dt.year if first_dt else parse_arxiv_id_year(arxiv_id)
    if year < MIN_YEAR:
        return None

    content_hash = hashlib.sha256(
        (title + "\n" + abstract).encode("utf-8")
    ).hexdigest()

    return {
        "arxiv_id": arxiv_id,
        "title": title,
        "abstract": abstract,
        "authors": " ".join((rec.get("authors") or "").split()),
        "all_categories": " ".join(sorted(all_categories)),
        "matched_categories": " ".join(matched),
        "first_submitted_at": first_dt.isoformat() if first_dt else None,
        "updated_at": latest_dt.isoformat() if latest_dt else rec.get("update_date"),
        "year": int(year),
        "doi": rec.get("doi"),
        "journal_ref": rec.get("journal-ref"),
        "license": rec.get("license"),
        "latest_version": f"v{len(versions)}" if versions else None,
        "content_hash": content_hash,
    }


def iter_filtered_records(path: str, skip_filtered: int = 0, limit: int | None = None):
    """
    Stream metadata snapshot line-by-line.
    `skip_filtered` counts records AFTER filtering, enabling resume on the same snapshot.
    """
    skipped = 0
    yielded = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue

            norm = normalize_record(rec)
            if norm is None:
                continue

            if skipped < skip_filtered:
                skipped += 1
                continue

            if limit is not None and yielded >= limit:
                break

            yield norm
            yielded += 1

## 5. Preview filter trước khi chạy full

Chỉ dùng khi `SKIP_EMBEDDING = False`.

In [13]:
if SKIP_EMBEDDING:
    print("SKIP_EMBEDDING=True -> skip filter preview")
else:
    preview = []
    for i, rec in enumerate(iter_filtered_records(SNAPSHOT_PATH, limit=5)):
        preview.append(rec)

    display(pd.DataFrame(preview)[
        ["arxiv_id", "title", "matched_categories", "year", "latest_version"]
    ])

SKIP_EMBEDDING=True -> skip filter preview


## 6. Load SPECTER2 paper encoder trên tối đa 2 GPU

Bỏ qua hoàn toàn khi `SKIP_EMBEDDING = True` (không load paper encoder, không load tokenizer). Smoke test query sẽ load tokenizer ở bước 12.

In [14]:
from transformers import AutoTokenizer
from adapters import AutoAdapterModel


def _assert_all_parameters_on(model, device, label):
    mismatched = [(name, str(p.device)) for name, p in model.named_parameters() if p.device != device]
    if mismatched:
        preview = mismatched[:5]
        raise RuntimeError(
            f"{label}: model parameters are split across devices. "
            f"Expected all on {device}; examples: {preview}"
        )


def load_paper_model(device):
    # IMPORTANT: load + activate adapter BEFORE model.to(device).
    # New adapter modules are created on CPU; moving the model afterwards guarantees
    # base model and adapter weights end up on the same CUDA device.
    model = AutoAdapterModel.from_pretrained(BASE_MODEL)
    adapter_name = model.load_adapter(
        PAPER_ADAPTER,
        source="hf",
        load_as="proximity",
    )
    model.active_adapters = adapter_name
    model.to(device)
    model.eval()

    _assert_all_parameters_on(model, device, f"paper encoder {device}")
    active = str(model.active_adapters)
    if "proximity" not in active:
        raise RuntimeError(f"Paper adapter not active on {device}: {active}")
    print(f"  active adapter on {device}: {active}")
    return model


models = []
tokenizer = None
if SKIP_EMBEDDING:
    print("SKIP_EMBEDDING=True -> skip SPECTER2 paper encoder and tokenizer")
    print("Query smoke test will load the tokenizer later.")
else:
    # Load tokenizer once on CPU; tokenize a full effective batch once, then split tensors by GPU.
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    for device in DEVICES:
        print(f"Loading paper encoder on {device} ...")
        models.append(load_paper_model(device))
    print(f"Loaded {len(models)} paper-encoder replica(s).")

SKIP_EMBEDDING=True -> skip SPECTER2 paper encoder and tokenizer
Query smoke test will load the tokenizer later.


## 7. Multi-GPU batch embedding + FP16 + OOM fallback

Bỏ qua khi `SKIP_EMBEDDING = True`.

In [15]:
def _slice_inputs(inputs, start, end):
    return {k: v[start:end] for k, v in inputs.items()}


@torch.inference_mode()
def _encode_inputs_on_device(model, device, cpu_inputs):
    """Encode one tensor slice on one device; recursively split on CUDA OOM."""
    n = next(iter(cpu_inputs.values())).shape[0]
    try:
        gpu_inputs = {
            k: v.to(device, non_blocking=True)
            for k, v in cpu_inputs.items()
        }
        autocast_enabled = (device.type == "cuda" and USE_FP16)
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=autocast_enabled,
        ):
            outputs = model(**gpu_inputs)
            emb = outputs.last_hidden_state[:, 0, :]

        # Canonical store/index vectors are L2-normalized.
        emb = F.normalize(emb.float(), p=2, dim=1)
        return emb.cpu().numpy()

    except RuntimeError as e:
        if device.type == "cuda" and "out of memory" in str(e).lower() and n > 1:
            print(f"[OOM on {device}] splitting sub-batch {n} -> {n//2} + {n-n//2}")
            torch.cuda.empty_cache()
            mid = n // 2
            left = _encode_inputs_on_device(model, device, _slice_inputs(cpu_inputs, 0, mid))
            right = _encode_inputs_on_device(model, device, _slice_inputs(cpu_inputs, mid, n))
            return np.concatenate([left, right], axis=0)
        raise


def embed_papers(records):
    """Preserves input order. Uses all configured devices in parallel."""
    texts = [
        r["title"] + tokenizer.sep_token + r["abstract"]
        for r in records
    ]
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
        return_token_type_ids=False,
    )
    if CUDA_AVAILABLE:
        # Pinned host memory makes non_blocking H2D copies useful.
        inputs = {k: v.pin_memory() for k, v in inputs.items()}

    n = len(records)
    workers = min(len(models), n)
    if workers == 1:
        return _encode_inputs_on_device(models[0], DEVICES[0], inputs)

    chunk = math.ceil(n / workers)
    jobs = []
    for worker_idx in range(workers):
        start = worker_idx * chunk
        end = min(n, start + chunk)
        if start >= end:
            break
        jobs.append((start, worker_idx, _slice_inputs(inputs, start, end)))

    pieces = []
    with ThreadPoolExecutor(max_workers=len(jobs)) as ex:
        future_map = {
            ex.submit(
                _encode_inputs_on_device,
                models[worker_idx],
                DEVICES[worker_idx],
                sub_inputs,
            ): start
            for start, worker_idx, sub_inputs in jobs
        }
        for fut in as_completed(future_map):
            pieces.append((future_map[fut], fut.result()))

    pieces.sort(key=lambda x: x[0])
    return np.concatenate([arr for _, arr in pieces], axis=0)


EFFECTIVE_BATCH_SIZE = (
    BATCH_SIZE_PER_GPU * len(DEVICES)
    if DEVICES[0].type == "cuda"
    else CPU_BATCH_SIZE
)
if SKIP_EMBEDDING:
    print("SKIP_EMBEDDING=True -> skip dual-GPU embed_papers()")
else:
    print("Effective batch size:", EFFECTIVE_BATCH_SIZE)

SKIP_EMBEDDING=True -> skip dual-GPU embed_papers()


## 8. Manifest + atomic checkpoint helpers

In [16]:
def atomic_write_json(path: Path, obj: dict):
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def new_manifest():
    return {
        "schema_version": 2,
        "config": CORPUS_CONFIG,
        "config_fingerprint": CONFIG_FINGERPRINT,
        "snapshot_fingerprint": SNAPSHOT_FINGERPRINT,
        "processed_papers": 0,
        "next_shard_idx": 0,
        "embedding_dim": None,
        "embedding_seconds": 0.0,
        "shards": [],
        "complete": False,
        "stop_reason": None,
        "index": None,
    }


def load_or_init_manifest():
    if SKIP_EMBEDDING:
        if not MANIFEST_PATH.exists():
            raise FileNotFoundError(
                f"SKIP_EMBEDDING=True nhưng không có {MANIFEST_PATH}. "
                "Hãy Add Input artifacts từ lần embed trước."
            )
        with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
            m = json.load(f)
        if m.get("config_fingerprint") != CONFIG_FINGERPRINT:
            raise RuntimeError(
                "Manifest config khác notebook config hiện tại. "
                "Hãy dùng OUTPUT_DIR mới hoặc xóa artifacts cũ."
            )
        if int(m.get("processed_papers") or 0) <= 0:
            raise RuntimeError("Manifest không có embeddings để tái sử dụng.")
        print(f"Loaded existing embeddings: {m['processed_papers']:,} filtered papers")
        return m

    if RESUME and MANIFEST_PATH.exists():
        with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
            m = json.load(f)

        if m.get("config_fingerprint") != CONFIG_FINGERPRINT:
            raise RuntimeError(
                "Manifest config khác notebook config hiện tại. "
                "Hãy dùng OUTPUT_DIR mới hoặc xóa artifacts cũ."
            )
        if (
            SNAPSHOT_FINGERPRINT is not None
            and m.get("snapshot_fingerprint") != SNAPSHOT_FINGERPRINT
        ):
            raise RuntimeError(
                "Snapshot fingerprint đã thay đổi. Resume-by-offset không còn an toàn. "
                "Hãy chạy incremental-update workflow hoặc dùng OUTPUT_DIR mới."
            )
        print(f"Resuming from {m['processed_papers']:,} filtered papers")
        return m

    m = new_manifest()
    atomic_write_json(MANIFEST_PATH, m)
    return m


manifest = load_or_init_manifest()
manifest

Loaded existing embeddings: 449,682 filtered papers


{'schema_version': 2,
 'config': {'target_categories': ['cs.AI',
   'cs.CL',
   'cs.CV',
   'cs.LG',
   'stat.ML'],
  'min_year': 2021,
  'base_model': 'allenai/specter2_base',
  'paper_adapter': 'allenai/specter2',
  'max_length': 512,
  'store_dtype': 'float16'},
 'config_fingerprint': '72350168c8d3570bb813d4260beb89e01502c34ccf77a4794100bd55fc7833bd',
 'snapshot_fingerprint': {'basename': 'arxiv-metadata-oai-snapshot.json',
  'size_bytes': 5496043808},
 'processed_papers': 449682,
 'next_shard_idx': 45,
 'embedding_dim': 768,
 'embedding_seconds': 1751.1228759890125,
 'shards': [{'shard_idx': 0,
   'row_start': 0,
   'row_end': 10048,
   'count': 10048,
   'metadata_file': 'metadata/metadata_part_00000.parquet',
   'embedding_file': 'embeddings/embeddings_part_00000.npy'},
  {'shard_idx': 1,
   'row_start': 10048,
   'row_end': 20096,
   'count': 10048,
   'metadata_file': 'metadata/metadata_part_00001.parquet',
   'embedding_file': 'embeddings/embeddings_part_00001.npy'},
  {'shard

## 9. Embedding pipeline: stream → dual GPU → sharded checkpoint

Bỏ qua khi `SKIP_EMBEDDING = True`. Manifest đã load từ artifacts có sẵn.

In [17]:
def flush_shard(buffer_records, buffer_embedding_chunks, manifest):
    if not buffer_records:
        return manifest

    shard_idx = manifest["next_shard_idx"]
    emb = np.concatenate(buffer_embedding_chunks, axis=0)
    assert len(buffer_records) == emb.shape[0]

    if manifest["embedding_dim"] is None:
        manifest["embedding_dim"] = int(emb.shape[1])
    assert emb.shape[1] == manifest["embedding_dim"]

    row_start = int(manifest["processed_papers"])
    row_end = row_start + len(buffer_records)

    meta_name = f"metadata_part_{shard_idx:05d}.parquet"
    emb_name = f"embeddings_part_{shard_idx:05d}.npy"
    meta_path = METADATA_DIR / meta_name
    emb_path = EMBEDDING_DIR / emb_name

    meta_tmp = meta_path.with_suffix(".parquet.tmp")
    emb_tmp = emb_path.with_suffix(".npy.tmp")

    # Metadata does not contain the vector itself.
    pd.DataFrame(buffer_records).to_parquet(meta_tmp, engine="pyarrow", index=False)
    with open(emb_tmp, "wb") as f:
        np.save(f, emb.astype(STORE_DTYPE, copy=False), allow_pickle=False)

    # Commit both files before manifest points to them.
    os.replace(meta_tmp, meta_path)
    os.replace(emb_tmp, emb_path)

    manifest["shards"].append({
        "shard_idx": shard_idx,
        "row_start": row_start,
        "row_end": row_end,
        "count": len(buffer_records),
        "metadata_file": str(meta_path.relative_to(OUTPUT_DIR)),
        "embedding_file": str(emb_path.relative_to(OUTPUT_DIR)),
    })
    manifest["processed_papers"] = row_end
    manifest["next_shard_idx"] = shard_idx + 1
    manifest["complete"] = False
    manifest["stop_reason"] = None
    atomic_write_json(MANIFEST_PATH, manifest)

    print(
        f"[checkpoint {shard_idx:05d}] {len(buffer_records):,} papers "
        f"| global rows [{row_start:,}, {row_end:,})"
    )
    return manifest


if SKIP_EMBEDDING:
    print(f"SKIP_EMBEDDING=True -> reuse {manifest['processed_papers']:,} embedded papers")
    print("Complete:", manifest.get("complete"), "| reason:", manifest.get("stop_reason"))
    print("Shards:", len(manifest.get("shards") or []))
    print("Embedding dim:", manifest.get("embedding_dim"))
else:
    processed_before_run = int(manifest["processed_papers"])
    remaining_limit = None
    if MAX_PAPERS is not None:
        remaining_limit = max(0, int(MAX_PAPERS) - processed_before_run)

    record_stream = iter_filtered_records(
        SNAPSHOT_PATH,
        skip_filtered=processed_before_run,
        limit=remaining_limit,
    )

    batch_records = []
    buffer_records = []
    buffer_embedding_chunks = []
    buffer_count = 0
    yielded_this_run = 0

    pbar = tqdm(
        initial=processed_before_run,
        total=MAX_PAPERS,
        desc="Embedding filtered papers",
        unit="paper",
    )

    for rec in record_stream:
        batch_records.append(rec)
        yielded_this_run += 1

        if len(batch_records) >= EFFECTIVE_BATCH_SIZE:
            t0 = time.perf_counter()
            embs = embed_papers(batch_records)
            manifest["embedding_seconds"] += time.perf_counter() - t0

            buffer_records.extend(batch_records)
            buffer_embedding_chunks.append(embs)
            buffer_count += len(batch_records)
            pbar.update(len(batch_records))
            batch_records = []

            if buffer_count >= SHARD_SIZE:
                manifest = flush_shard(buffer_records, buffer_embedding_chunks, manifest)
                buffer_records, buffer_embedding_chunks, buffer_count = [], [], 0

    # Final partial batch.
    if batch_records:
        t0 = time.perf_counter()
        embs = embed_papers(batch_records)
        manifest["embedding_seconds"] += time.perf_counter() - t0
        buffer_records.extend(batch_records)
        buffer_embedding_chunks.append(embs)
        buffer_count += len(batch_records)
        pbar.update(len(batch_records))

    # Commit final partial shard.
    if buffer_records:
        manifest = flush_shard(buffer_records, buffer_embedding_chunks, manifest)

    pbar.close()

    # Determine whether the filtered stream was exhausted or intentionally truncated.
    if remaining_limit is None:
        manifest["complete"] = True
        manifest["stop_reason"] = "stream_exhausted"
    elif yielded_this_run < remaining_limit:
        manifest["complete"] = True
        manifest["stop_reason"] = "stream_exhausted"
    else:
        manifest["complete"] = False
        manifest["stop_reason"] = "max_papers_reached"

    atomic_write_json(MANIFEST_PATH, manifest)

    print(f"Processed total: {manifest['processed_papers']:,}")
    print("Complete:", manifest["complete"], "| reason:", manifest["stop_reason"])
    if manifest["embedding_seconds"] > 0:
        print(
            "Approx embedding throughput:",
            round(manifest["processed_papers"] / manifest["embedding_seconds"], 2),
            "papers/s (compute only)"
        )

SKIP_EMBEDDING=True -> reuse 449,682 embedded papers
Complete: True | reason: stream_exhausted
Shards: 45
Embedding dim: 768


## 10. Validate checkpoint integrity + embedding norms

In [18]:
def validate_artifacts(manifest, sample_per_shard=32):
    total = 0
    dims = set()
    norm_samples = []

    for shard in manifest["shards"]:
        meta_path = OUTPUT_DIR / shard["metadata_file"]
        emb_path = OUTPUT_DIR / shard["embedding_file"]
        assert meta_path.exists(), meta_path
        assert emb_path.exists(), emb_path

        df = pd.read_parquet(meta_path, columns=["arxiv_id"])
        emb = np.load(emb_path, mmap_mode="r", allow_pickle=False)

        assert len(df) == shard["count"] == emb.shape[0]
        dims.add(int(emb.shape[1]))
        total += len(df)

        take = min(sample_per_shard, len(emb))
        if take:
            sample = np.asarray(emb[:take], dtype=np.float32)
            norm_samples.extend(np.linalg.norm(sample, axis=1).tolist())

    assert total == manifest["processed_papers"]
    assert len(dims) <= 1

    print("Validated papers:", f"{total:,}")
    print("Embedding dims:", dims)
    if norm_samples:
        print(
            "Sample L2 norms: min/mean/max =",
            round(float(np.min(norm_samples)), 5),
            round(float(np.mean(norm_samples)), 5),
            round(float(np.max(norm_samples)), 5),
        )

validate_artifacts(manifest)

Validated papers: 449,682
Embedding dims: {768}
Sample L2 norms: min/mean/max = 0.99982 1.0 1.00018


## 11. Build exact FAISS cosine index incrementally from shards

In [19]:
import faiss

assert manifest["processed_papers"] > 0, "No embeddings available."
dim = int(manifest["embedding_dim"])


def _existing_index_is_valid():
    info = manifest.get("index") or {}
    return (
        INDEX_PATH.exists()
        and info.get("type") == "IndexFlatIP"
        and int(info.get("ntotal", -1)) == int(manifest["processed_papers"])
        and int(info.get("dim", -1)) == dim
    )


if _existing_index_is_valid():
    index = faiss.read_index(str(INDEX_PATH))
    assert index.ntotal == manifest["processed_papers"]
    assert index.d == dim
    print("Reusing existing FAISS index:", INDEX_PATH)
else:
    # Exact baseline. Because vectors are L2-normalized, inner product == cosine similarity.
    index = faiss.IndexFlatIP(dim)

    for shard in tqdm(manifest["shards"], desc="Building FAISS", unit="shard"):
        emb_path = OUTPUT_DIR / shard["embedding_file"]
        emb = np.load(emb_path, mmap_mode="r", allow_pickle=False)

        # FAISS expects float32. Load one shard only, so peak RAM remains bounded.
        vec = np.asarray(emb, dtype=np.float32)
        # Re-normalize defensively after float16 storage round-trip.
        faiss.normalize_L2(vec)
        index.add(vec)

    assert index.ntotal == manifest["processed_papers"]
    faiss.write_index(index, str(INDEX_PATH))

    manifest["index"] = {
        "type": "IndexFlatIP",
        "metric": "cosine_via_normalized_inner_product",
        "file": str(INDEX_PATH.relative_to(OUTPUT_DIR)),
        "ntotal": int(index.ntotal),
        "dim": dim,
    }
    atomic_write_json(MANIFEST_PATH, manifest)

print("FAISS index:", INDEX_PATH)
print("Vectors:", f"{index.ntotal:,}", "| dim:", dim)


Reusing existing FAISS index: /kaggle/input/datasets/thanhtruchhong/specter2-artifacts/specter2_artifacts/index/papers_flatip.faiss
FAISS index: /kaggle/input/datasets/thanhtruchhong/specter2-artifacts/specter2_artifacts/index/papers_flatip.faiss
Vectors: 449,682 | dim: 768


## 12. Retrieval smoke test bằng SPECTER2 adhoc-query

Đây là test integration quan trọng hơn việc tính cosine giữa hai paper ngẫu nhiên:
- query ngắn được encode bằng `allenai/specter2_adhoc_query`;
- candidates trong index đã được encode bằng proximity adapter;
- FAISS trả top-k;
- global vector row được map ngược về metadata shard.

In [20]:
# Use a dedicated query encoder instead of loading the query adapter into a paper
# encoder that is already on CUDA. Loading an adapter after model.to(cuda) creates
# new adapter weights on CPU in current adapters versions and can cause a device mismatch.
if tokenizer is None:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

query_device = (
    torch.device(f"cuda:{min(QUERY_GPU, GPU_COUNT - 1)}")
    if CUDA_AVAILABLE and GPU_COUNT > 0
    else torch.device("cpu")
)


def load_query_model(device):
    model = AutoAdapterModel.from_pretrained(BASE_MODEL)
    adapter_name = model.load_adapter(
        QUERY_ADAPTER,
        source="hf",
        load_as="adhoc_query",
    )
    model.active_adapters = adapter_name
    # Move AFTER the adapter exists so both base + adapter weights move together.
    model.to(device)
    model.eval()

    _assert_all_parameters_on(model, device, f"query encoder {device}")
    active = str(model.active_adapters)
    if "adhoc_query" not in active:
        raise RuntimeError(f"Query adapter not active on {device}: {active}")
    print(f"Query encoder: {device} | active adapter: {active}")
    return model


query_model = load_query_model(query_device)


@torch.inference_mode()
def embed_query(query: str) -> np.ndarray:
    inputs = tokenizer(
        [query],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
        return_token_type_ids=False,
    ).to(query_device)

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(query_device.type == "cuda" and USE_FP16),
    ):
        out = query_model(**inputs)
        q = out.last_hidden_state[:, 0, :]

    q = F.normalize(q.float(), p=2, dim=1)
    return q.cpu().numpy().astype(np.float32)


def lookup_global_rows(global_ids):
    wanted = [int(x) for x in global_ids]
    result = {}

    for shard in manifest["shards"]:
        s, e = shard["row_start"], shard["row_end"]
        local_wanted = [g for g in wanted if s <= g < e]
        if not local_wanted:
            continue

        df = pd.read_parquet(
            OUTPUT_DIR / shard["metadata_file"],
            columns=["arxiv_id", "title", "matched_categories", "year"],
        )
        for g in local_wanted:
            row = df.iloc[g - s]
            result[g] = row.to_dict()

    missing = [g for g in wanted if g not in result]
    if missing:
        raise KeyError(f"Could not map FAISS ids back to metadata: {missing[:10]}")
    return [result[g] for g in wanted]


TEST_QUERY = "dataset pruning and data subset selection for deep neural networks"
TOP_K = min(10, index.ntotal)
q = embed_query(TEST_QUERY)
assert q.shape == (1, dim)
assert np.isfinite(q).all()
assert abs(float(np.linalg.norm(q[0])) - 1.0) < 1e-4

scores, ids = index.search(q, TOP_K)
rows = lookup_global_rows(ids[0])

results = pd.DataFrame(rows)
results.insert(0, "score", scores[0])
results.insert(0, "rank", np.arange(1, len(results) + 1))
print("Query:", TEST_QUERY)
results


tokenizer_config.json:   0%|          | 0.00/453 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

pytorch_adapter.bin:   0%|          | 0.00/3.59M [00:00<?, ?B/s]

Query encoder: cuda:0 | active adapter: Stack[adhoc_query]
Query: dataset pruning and data subset selection for deep neural networks


,rank,score,arxiv_id,title,matched_categories,year
0,1,0.852230,2606.21916,"Data Pruning: Redundant, Problematic, and Inte...",cs.LG,2026
1,2,0.848332,2312.05599,Not All Data Matters: An End-to-End Adaptive D...,cs.AI cs.LG,2023
2,3,0.831760,2205.09329,Dataset Pruning: Reducing Training Data by Exa...,cs.LG,2022
3,4,0.822949,2603.26138,PruneFuse: Efficient Data Selection via Weight...,cs.CV cs.LG,2026
4,5,0.822419,2205.15731,ViNNPruner: Visual Interactive Pruning for Dee...,cs.LG,2022
5,6,0.822386,2305.18424,Repeated Random Sampling for Minimizing the Ti...,cs.CV cs.LG,2023
6,7,0.819424,2501.01118,Pruning-based Data Selection and Network Fusio...,cs.AI cs.LG,2025
7,8,0.817568,2406.03057,BWS: Best Window Selection Based on Sample Sco...,cs.LG stat.ML,2024
8,9,0.816592,2505.07411,ICE-Pruning: An Iterative Cost-Efficient Pruni...,cs.CV cs.LG stat.ML,2025
9,10,0.813860,2303.00566,Structured Pruning for Deep Convolutional Neur...,cs.CV,2023


## 13. Corpus diagnostics

In [21]:
# Metadata-only aggregation. Reads one metadata shard at a time.
by_year = {}
by_category = {}

for shard in manifest["shards"]:
    df = pd.read_parquet(
        OUTPUT_DIR / shard["metadata_file"],
        columns=["year", "matched_categories"],
    )

    for year, count in df["year"].value_counts().items():
        by_year[int(year)] = by_year.get(int(year), 0) + int(count)

    for cats in df["matched_categories"]:
        for cat in str(cats).split():
            by_category[cat] = by_category.get(cat, 0) + 1

print("Papers by year:")
print(pd.Series(by_year).sort_index())
print("\nMatched-category counts (multi-label, so sum can exceed paper count):")
print(pd.Series(by_category).sort_values(ascending=False))

Papers by year:
2021     48301
2022     53340
2023     66909
2024     86662
2025    106848
2026     87622
dtype: int64

Matched-category counts (multi-label, so sum can exceed paper count):
cs.LG      210062
cs.AI      167493
cs.CV      152490
cs.CL       93269
stat.ML     32564
dtype: int64


## 14. Save a compact run summary

In [26]:
summary = {
    "processed_papers": manifest["processed_papers"],
    "complete": manifest["complete"],
    "stop_reason": manifest["stop_reason"],
    "embedding_dim": manifest["embedding_dim"],
    "num_shards": len(manifest["shards"]),
    "devices": [str(d) for d in DEVICES],
    "gpu_names": [
        torch.cuda.get_device_name(i) for i in range(NUM_DEVICES)
    ] if CUDA_AVAILABLE else [],
    "effective_batch_size": EFFECTIVE_BATCH_SIZE,
    "embedding_seconds": manifest["embedding_seconds"],
    "approx_compute_papers_per_second": (
        manifest["processed_papers"] / manifest["embedding_seconds"]
        if manifest["embedding_seconds"] > 0 else None
    ),
    "index": manifest.get("index"),
    "config_fingerprint": manifest["config_fingerprint"],
}
OUTPUT_DIR = Path("/kaggle/working")
summary_path = OUTPUT_DIR / "run_summary.json"
atomic_write_json(summary_path, summary)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("Artifacts root:", OUTPUT_DIR)

{
  "processed_papers": 449682,
  "complete": true,
  "stop_reason": "stream_exhausted",
  "embedding_dim": 768,
  "num_shards": 45,
  "devices": [
    "cuda:0",
    "cuda:1"
  ],
  "gpu_names": [
    "Tesla T4",
    "Tesla T4"
  ],
  "effective_batch_size": 64,
  "embedding_seconds": 1751.1228759890125,
  "approx_compute_papers_per_second": 256.79637115472275,
  "index": {
    "type": "IndexFlatIP",
    "metric": "cosine_via_normalized_inner_product",
    "file": "index/papers_flatip.faiss",
    "ntotal": 449682,
    "dim": 768
  },
  "config_fingerprint": "72350168c8d3570bb813d4260beb89e01502c34ccf77a4794100bd55fc7833bd"
}
Artifacts root: /kaggle/working


## 15. Ghi chú vận hành

### SKIP_EMBEDDING
Đặt `SKIP_EMBEDDING = True` (mặc định hiện tại) để **không embed lại**. Notebook sẽ:
1. tìm `manifest.json` trong `OUTPUT_DIR` hoặc `/kaggle/input/*/`;
2. symlink `embeddings/` + `metadata/` vào `/kaggle/working/specter2_artifacts`;
3. validate shards, build FAISS, chạy smoke test query.

Session mới trên Kaggle: **Add Input** output lần embed trước. Nếu auto-detect sai, set `EMBEDDING_INPUT_DIR = Path("/kaggle/input/<dataset-slug>")` (thư mục chứa `manifest.json`, hoặc `.../specter2_artifacts`).

Đặt `SKIP_EMBEDDING = False` để chạy lại paper encoder + pipeline embed.

### Resume
`manifest.json` chỉ resume theo offset khi **snapshot fingerprint và corpus config không đổi**. Nếu snapshot arXiv đổi version, không nên skip theo row offset; hãy dùng incremental updater dựa trên `arxiv_id + content_hash`.

### MAX_PAPERS
- `5000`: smoke test nhanh.
- `None`: full filtered corpus.
- Thay đổi `MAX_PAPERS` không đổi corpus fingerprint, nên có thể tiếp tục từ checkpoint nếu thư mục artifacts còn tồn tại.

### 2 GPU
Notebook dùng hai model replicas và parallel sub-batches nếu runtime có >=2 CUDA devices. Nếu Kaggle chỉ cấp 1 GPU, code tự fallback.

### Storage
- `.npy` embedding shard dùng `float16` để giảm disk.
- FAISS `IndexFlatIP` giữ `float32`, ưu tiên exact retrieval cho baseline/evaluation.
- Nếu index quá lớn/latency cao, benchmark HNSW hoặc IVF/PQ ở notebook riêng thay vì thay baseline ngay.

### Online retrieval sau này
Không dùng proximity adapter cho short query. Online query encoder phải dùng `allenai/specter2_adhoc_query`; candidate-paper embeddings trong index dùng proximity adapter như notebook này.

### Incremental update
Notebook này implement **resume cùng snapshot**. Incremental update giữa hai snapshot nên là job riêng để xử lý cả new paper và changed abstract/version một cách đúng đắn.

### Adapter/device safety
- Load adapter **before** `model.to(device)`; otherwise newly added adapter modules may remain on CPU.
- Paper and query encoders are separate models. Paper index uses `proximity`; short user queries use `adhoc_query`.
- Notebook asserts all parameters are on the expected device and prints the active adapter before inference.

### Kaggle package safety
Do not `pip install -U pandas pyarrow ...` inside the notebook. Kaggle already pins these packages for RAPIDS/BigFrames; unnecessary upgrades can cause dependency conflicts.
